# Process lineage Sigma hunt

Trace the **process lineage** behind Sigma rule hits and rank **rare** network / user / URL activity — a notebook port of [MohitDabas/sigmalineage-mcp](https://github.com/MohitDabas/sigmalineage-mcp).

**Data plane:** Chainsaw and raw `.evtx` parsing cannot run in Pyodide, so Sigma hits and the parsed Windows process / telemetry events are **precomputed** and hosted as CSVs in [notebook-app-example-data](https://github.com/michaelhyatt/notebook-app-example-data). They load through **Cribl Search** [`externaldata`](https://docs.cribl.io/search/externaldata/) (same pattern as the Anomaly and Malware Hash notebooks) — not via Pyodide `fetch`, and **not** through `config/proxies.yml`. Lineage tracing (parent → child) and the rarity baseline run **in-kernel** with **pandas** + **networkx**.

**Run All** works out of the box: no auth, no proxy change. Open this copy from **Welcome → Examples** (not an old saved tab).

## What you will do

| Step | Question it answers |
|------|---------------------|
| Load process events | What process-creation events (Sysmon EID 1 / Security 4688) did we collect? |
| Load Sigma hits | Which rules fired, and on which processes? |
| Build index | How do parent → child links resolve (ProcessGuid, else PID + time)? |
| Trace lineage | What is the full ancestry (kill chain) behind each hit? |
| Rarity baseline | Which port / user / URL tuples are anomalously rare? |

```
windows_process_events.csv ──externaldata──► proc_events ─┐
sigma_hits.csv             ──externaldata──► sigma_hits ──┤─► lineage trees + networkx kill-chain
windows_telemetry_events.csv ──externaldata──► telemetry ─┘─► rarity baseline (3 tuple families)
```

**Related:** `Malware_Hash_Threat_Hunt.ipynb`, `Threat_Hunting_Playbook.ipynb`, `Cribl_Search_Examples.ipynb`.

## Prerequisites

1. Hosted Cribl app with **Cribl Search** and `%%cribl_search`.
2. **No auth / no `proxies.yml` change** — Search workers HTTP GET the raw GitHub CSVs (not the app pack `/data/` path).
3. **Optional overrides:** `PROCESS_LINEAGE_EVENTS_URL`, `PROCESS_LINEAGE_HITS_URL`, `PROCESS_LINEAGE_TELEMETRY_URL`.
4. Run the **Setup** code cell first (**Run All** does this automatically) — it defines all helper functions.
5. `networkx` ships in the Pyodide lockfile and loads on `import networkx` (no `micropip`).

## Setup — URLs and lineage / rarity helpers

Run this first. Defines the dataset URLs and the pure-Python helpers ported from
[sigmalineage-mcp](https://github.com/MohitDabas/sigmalineage-mcp) (`sigma_lineage.py`, `services/rarity.py`):
process indexing + parent resolution, `trace_lineage`, `render_lineage_tree`, and
`compute_rare_events_with_baseline`.

In [ ]:
# Example datasets: https://github.com/michaelhyatt/notebook-app-example-data
# Keep URLs in sync with src/domain/exampleDataUrls.ts
import io
import os
from collections import Counter

import pandas as pd

_EXAMPLE_BASE = "https://raw.githubusercontent.com/michaelhyatt/notebook-app-example-data/main"
PROCESS_EVENTS_URL = (os.environ.get("PROCESS_LINEAGE_EVENTS_URL") or "").strip() or f"{_EXAMPLE_BASE}/process-lineage/windows_process_events.csv"
SIGMA_HITS_URL = (os.environ.get("PROCESS_LINEAGE_HITS_URL") or "").strip() or f"{_EXAMPLE_BASE}/process-lineage/sigma_hits.csv"
TELEMETRY_URL = (os.environ.get("PROCESS_LINEAGE_TELEMETRY_URL") or "").strip() or f"{_EXAMPLE_BASE}/process-lineage/windows_telemetry_events.csv"

MAX_LINEAGE_LEVELS = 5

print(f"Process events: {PROCESS_EVENTS_URL}")
print(f"Sigma hits:     {SIGMA_HITS_URL}")
print(f"Telemetry:      {TELEMETRY_URL}")

_MISSING = {"", "nan", "none", "-"}


def _clean(val):
    s = str(val).strip().strip('"')
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]  # tidy float-typed ports / ids (e.g. 8443.0 -> 8443)
    return "" if s.lower() in _MISSING else s


def _pick_col(frame, *names):
    keys = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n.lower() in keys:
            return keys[n.lower()]
    return None


def _csv_lines_from_raw(frame):
    raw_col = _pick_col(frame, "_raw", "event")
    if raw_col is None:
        return []
    lines = []
    for blob in frame[raw_col].astype(str):
        lines.extend(blob.splitlines())
    return [ln for ln in lines if ln.strip()]


def materialize_search_csv(frame, *, expect):
    """Clean DataFrame from a %%cribl_search externaldata result.

    Uses typed columns when Search returns them; otherwise rebuilds from the
    _raw / event blob (same fallback as the Anomaly / Malware notebooks).
    """
    if _pick_col(frame, *expect) is not None:
        out = frame.copy()
    else:
        lines = _csv_lines_from_raw(frame)
        if not lines:
            raise ValueError(f"No typed columns {expect} and no _raw/event rows; got {list(frame.columns)}")
        out = pd.read_csv(io.StringIO("\n".join(lines)))
    out.columns = [str(c).strip().strip('"') for c in out.columns]
    for col in out.columns:
        if out[col].dtype == object:
            out[col] = out[col].astype(str).str.strip().str.strip('"')
    drop = [c for c in out.columns if str(c).startswith("_") or str(c).lower() in {"source", "sourcetype", "index", "cribl_pipe"}]
    return out.drop(columns=drop, errors="ignore")


def parse_pid(val):
    s = _clean(val)
    if not s:
        return 0
    try:
        return int(s, 16) if s.lower().startswith("0x") else int(float(s))
    except ValueError:
        return 0


def parse_time(val):
    return pd.to_datetime(_clean(val) or None, errors="coerce", utc=True)


def _find_parent_key(processes, computer, parent_pid, child_time_str):
    """Nearest earlier same-PID process on the same host (sigma_lineage.find_parent_key)."""
    child_time = parse_time(child_time_str)
    best_key, min_diff = None, None
    for key, proc in processes.items():
        if proc["computer"] == computer and proc["pid"] == parent_pid:
            if pd.isna(child_time):
                return key
            proc_time = parse_time(proc["time"])
            if pd.isna(proc_time):
                continue
            diff = (child_time - proc_time).total_seconds()
            if diff >= 0 and (min_diff is None or diff < min_diff):
                min_diff, best_key = diff, key
    return best_key


def build_process_index(proc_df):
    """Normalize process-creation rows into {key: process} with resolved parent links.

    Prefer ProcessGuid as key / parent link; else match (Computer, ParentProcessId)
    to the nearest earlier event time.
    """
    col = {n: _pick_col(proc_df, n) for n in (
        "ProcessGuid", "ProcessId", "Image", "CommandLine",
        "ParentProcessGuid", "ParentProcessId", "ParentImage",
        "Computer", "User", "UtcTime",
    )}

    def get(row, name):
        c = col[name]
        return row[c] if c is not None and c in row else None

    processes = {}
    for _, row in proc_df.iterrows():
        guid = _clean(get(row, "ProcessGuid")) or None
        pid = parse_pid(get(row, "ProcessId"))
        computer = _clean(get(row, "Computer"))
        t = _clean(get(row, "UtcTime"))
        key = guid or f"{computer}:{pid}:{t}"
        processes[key] = {
            "guid": guid,
            "pid": pid,
            "image": _clean(get(row, "Image")) or "-",
            "command_line": _clean(get(row, "CommandLine")) or "-",
            "parent_guid": _clean(get(row, "ParentProcessGuid")) or None,
            "parent_pid": parse_pid(get(row, "ParentProcessId")),
            "parent_image": _clean(get(row, "ParentImage")) or "-",
            "computer": computer,
            "user": _clean(get(row, "User")) or "-",
            "time": t,
        }

    guid_index = {p["guid"]: k for k, p in processes.items() if p["guid"]}
    for proc in processes.values():
        if proc["parent_guid"] and proc["parent_guid"] in guid_index:
            proc["parent_key"] = guid_index[proc["parent_guid"]]
        elif proc["parent_pid"]:
            proc["parent_key"] = _find_parent_key(processes, proc["computer"], proc["parent_pid"], proc["time"])
        else:
            proc["parent_key"] = None
    return processes


def _hit_start_key(processes, guid, pid, computer, time_str):
    if guid and guid in processes:
        return guid
    guid_index = {p["guid"]: k for k, p in processes.items() if p["guid"]}
    if guid and guid in guid_index:
        return guid_index[guid]
    key = _find_parent_key(processes, computer, pid, time_str)
    if key:
        return key
    for k, p in processes.items():
        if p["computer"] == computer and p["pid"] == pid:
            return k
    return None


def trace_lineage(processes, hit, levels=None):
    """Walk parents from a Sigma hit up to `levels` ancestors (level 0 = the hit)."""
    levels = MAX_LINEAGE_LEVELS if levels is None else levels
    guid = _clean(hit.get("ProcessGuid")) or None
    pid = parse_pid(hit.get("ProcessId"))
    computer = _clean(hit.get("Computer"))
    time_str = _clean(hit.get("UtcTime"))
    current = _hit_start_key(processes, guid, pid, computer, time_str)
    path, visited, depth = [], set(), 0
    while current and current not in visited and depth <= levels:
        visited.add(current)
        proc = processes.get(current)
        if not proc:
            break
        path.append({"level": depth, **proc})
        current = proc.get("parent_key")
        depth += 1
    return {
        "rule_name": hit.get("rule_name", "Unknown Rule"),
        "severity": _clean(hit.get("level")),
        "computer": computer,
        "timestamp": time_str,
        "path": path,
    }


def _short_image(image):
    return image.replace("/", "\\").split("\\")[-1]


def render_lineage_tree(lineage):
    header = f"[{(lineage['severity'] or '?').upper()}] {lineage['rule_name']}  @ {lineage['computer']}  {lineage['timestamp']}"
    lines = [header]
    for depth, node in enumerate(reversed(lineage["path"])):  # root -> hit
        indent = "   " * depth
        prefix = indent if depth == 0 else indent[:-3] + "└─ "
        marker = "  (HIT)" if node["level"] == 0 else ""
        lines.append(f"{prefix}{_short_image(node['image'])} (pid {node['pid']}){marker}  {node['command_line']}")
    return "\n".join(lines)


def extract_tuples(rows):
    """Three rarity tuple families (services/rarity.py extract_tuples)."""
    families = {"process_dst_port_protocol": [], "user_channel_event_id": [], "url_host_process": []}
    for row in rows:
        def gv(*names):
            for n in names:
                if n in row:
                    v = _clean(row[n])
                    if v:
                        return v
            return ""
        image = gv("Image", "ProcessName", "NewProcessName")
        dport = gv("DestinationPort", "DestPort")
        proto = gv("Protocol")
        user = gv("User", "SubjectUserName", "TargetUserName")
        channel = gv("Channel")
        eid = gv("EventID")
        url = gv("url", "Url")
        host = gv("Computer", "DestinationHostname")
        if image or dport or proto:
            families["process_dst_port_protocol"].append((image or "-", dport or "-", proto or "-"))
        if user or channel or eid:
            families["user_channel_event_id"].append((user or "-", channel or "-", eid or "-"))
        if url or host or image:
            families["url_host_process"].append((url or "-", host or "-", image or "-"))
    return families


def compute_rare_events_with_baseline(target_df, baseline_df=None, max_results=25, max_baseline_count=2):
    """Rare tuples vs a baseline (defaults to self). rarity_score = 1/(baseline_count+1)."""
    target_rows = target_df.to_dict("records")
    baseline_rows = target_rows if baseline_df is None else baseline_df.to_dict("records")
    tf, bf = extract_tuples(target_rows), extract_tuples(baseline_rows)
    results = {}
    for fam in ("process_dst_port_protocol", "user_channel_event_id", "url_host_process"):
        tc, bc = Counter(tf[fam]), Counter(bf[fam])
        rows = [
            {
                "tuple": " | ".join(tup),
                "target_count": t_count,
                "baseline_count": bc[tup],
                "rarity_score": round(1.0 / (bc[tup] + 1.0), 4),
            }
            for tup, t_count in tc.items()
            if bc[tup] <= max_baseline_count
        ]
        rows.sort(key=lambda r: (r["rarity_score"], r["target_count"], -r["baseline_count"]), reverse=True)
        results[fam] = rows[:max_results]
    return results


print("Setup complete: helpers defined.")

## A) Load process-creation events (`externaldata`)

Fetches `windows_process_events.csv` (Sysmon EID 1 + Security EID 4688) through Search
[`externaldata`](https://docs.cribl.io/search/externaldata/).

In [ ]:
%%cribl_search var=proc_events_raw lang=kql limit=5000 preview=true earliest=-1h latest=now timeout=300 template=on
externaldata
[
  "{{ PROCESS_EVENTS_URL }}"
]
with(
  datatype="CSV Datatypes"
)

In [ ]:
proc_events = materialize_search_csv(proc_events_raw, expect=("ProcessId", "Image"))
_host_col = _pick_col(proc_events, "Computer")
print(f"Process events: {len(proc_events):,}  |  hosts: {sorted(proc_events[_host_col].unique()) if _host_col else 'n/a'}")
proc_events.head(12)

## B) Load Sigma hits (`externaldata`)

Precomputed Chainsaw/Sigma hits — each joins to a process event on `ProcessGuid` (else `Computer` + `ProcessId` + `UtcTime`).

In [ ]:
%%cribl_search var=sigma_hits_raw lang=kql limit=2000 preview=true earliest=-1h latest=now timeout=300 template=on
externaldata
[
  "{{ SIGMA_HITS_URL }}"
]
with(
  datatype="CSV Datatypes"
)

In [ ]:
sigma_hits = materialize_search_csv(sigma_hits_raw, expect=("rule_name", "ProcessId"))
print(f"Sigma hits: {len(sigma_hits):,}")
sigma_hits[[c for c in ["rule_name", "level", "Computer", "ProcessId", "Image"] if c in sigma_hits.columns]]

## C) Build the process index and resolve parents

Prefers `ProcessGuid`; the EID 4688 rows (no GUID) fall back to `(Computer, ParentProcessId)` matched to the nearest earlier event time.

In [ ]:
processes = build_process_index(proc_events)
linked = sum(1 for p in processes.values() if p.get("parent_key"))
print(f"Indexed {len(processes):,} processes; {linked:,} linked to a parent.")

## D) Trace lineage for each hit

Walks up to `MAX_LINEAGE_LEVELS` (5) ancestors and prints the kill chain (root → `(HIT)`).

In [ ]:
lineages = [trace_lineage(processes, row) for row in sigma_hits.to_dict("records")]
for lin in lineages:
    print(render_lineage_tree(lin))
    print()
if not any(lin["path"] for lin in lineages):
    raise ValueError("No lineage resolved for any Sigma hit — confirm §A/§B loaded, then Run All.")

## E) Visualize the kill chain (`networkx`)

Draws the merged parent → child graph across all hits; **red** nodes are Sigma hits. `networkx` loads from the Pyodide lockfile on import.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

G = nx.DiGraph()
hit_nodes = set()
for lin in lineages:
    path = lin["path"]
    for node in path:
        nid = f"{node['computer']}:{node['pid']}"
        G.add_node(nid, label=f"{_short_image(node['image'])}\n(pid {node['pid']})")
    for child, parent in zip(path, path[1:]):  # edge parent -> child
        G.add_edge(f"{parent['computer']}:{parent['pid']}", f"{child['computer']}:{child['pid']}")
    if path:
        hit_nodes.add(f"{path[0]['computer']}:{path[0]['pid']}")

for layer, gen in enumerate(nx.topological_generations(G)):
    for n in gen:
        G.nodes[n]["layer"] = layer
pos = nx.multipartite_layout(G, subset_key="layer", align="horizontal")
pos = {n: (x, -y) for n, (x, y) in pos.items()}  # root on top

fig, ax = plt.subplots(figsize=(11, 7))
colors = ["#c44e52" if n in hit_nodes else "#4c72b0" for n in G.nodes]
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", edge_color="#999999", width=1.4, node_size=3000)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=3000, edgecolors="white", linewidths=1.5)
nx.draw_networkx_labels(G, pos, labels={n: G.nodes[n]["label"] for n in G.nodes}, ax=ax, font_size=8, font_color="white")
ax.set_title("Process lineage kill-chain (red = Sigma hit)")
ax.axis("off")
plt.tight_layout()
plt.show()

## F) Rarity baseline — surface anomalous tuples

Ranks rare combinations across three families (self-baseline; `rarity_score = 1/(baseline_count + 1)`).

In [ ]:
%%cribl_search var=telemetry_raw lang=kql limit=10000 preview=true earliest=-1h latest=now timeout=300 template=on
externaldata
[
  "{{ TELEMETRY_URL }}"
]
with(
  datatype="CSV Datatypes"
)

In [ ]:
telemetry = materialize_search_csv(telemetry_raw, expect=("EventID", "Computer"))
print(f"Telemetry rows: {len(telemetry):,}")
rare = compute_rare_events_with_baseline(telemetry)
for fam, rows in rare.items():
    print(f"\n=== {fam} (top {len(rows)}) ===")
    for r in rows[:5]:
        print(f"  score={r['rarity_score']:.3f}  seen={r['target_count']}x  {r['tuple']}")

## G) Chart rarity scores

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (fam, rows) in zip(axes, rare.items()):
    rows = rows[:6]
    labels = [(r["tuple"] if len(r["tuple"]) <= 34 else r["tuple"][:31] + "…") for r in rows]
    scores = [r["rarity_score"] for r in rows]
    ax.barh(range(len(rows)), scores, color="#dd8452")
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.invert_yaxis()
    ax.set_title(fam, fontsize=9)
    ax.set_xlabel("rarity score")
plt.tight_layout()
plt.show()

## Interpretation

- **Lineage is the story of a hit.** A Sigma match on `powershell.exe` or `rundll32.exe` means little until you see it was spawned by `WmiPrvSE.exe` (WMI lateral movement) rather than `explorer.exe` — the parent → child chain confirms the kill chain.
- **Parent resolution** uses `ProcessGuid` when present (Sysmon EID 1) and falls back to `(Computer, ParentProcessId)` + nearest earlier time for Security EID 4688 rows without a GUID.
- **Rarity surfaces the unusual.** A single `rundll32.exe → :4444/tcp` connection, a lone `svc_backup` interactive logon, or a one-off C2 URL stand out against the noisy baseline — high `rarity_score`, low `baseline_count`.
- **Production:** run Chainsaw + Sigma over real `.evtx` to generate hits, ship the parsed events + hits into Cribl Search, and point the URL overrides at your own hosted CSVs.

## Troubleshooting

| Symptom | Fix |
|---------|-----|
| No rows / wrong columns from `externaldata` | `materialize_search_csv` rebuilds from `_raw`/`event` when typed columns are missing; re-run §A/§B. If Search rejects `CSV Datatypes`, try `CSV`. |
| `No lineage resolved` | §A and §B must succeed first; then Run All. Check the hits' `ProcessGuid`/`ProcessId` match the process events. |
| `ModuleNotFoundError: networkx` | Re-run §E — `networkx` loads from the Pyodide lockfile on import (first import can take a moment). |
| Helpers undefined (`trace_lineage` …) | Re-open the example from **Welcome** (stale tab) and run **Setup** first. |
| Stale notebook tab | Close tab → **Welcome** → **Process Lineage Sigma Hunt** → **Open example**. |